# 07 — Assess relative productivity performance

Compare each pixel's NDVI metric with the maximum baseline metric within its ESA WorldCover class in the study area.


## Inputs, outputs, and execution order

**Inputs:** Annual NDVI rasters from 04, the study boundary, and `esa_worldcover_2021` from the DE Africa data cube.

**Outputs:** Baseline, assessment-year, and reporting-window mean/ratio/class GeoTIFFs under `performance/`.

Run cells from top to bottom in a fresh kernel. Open this notebook beside `workflow_config.json` and `workflow_support.py`. The configuration uses Mion study settings; the original boundary and final criteria table were not supplied. See the README for the small example and for adapting the workflow.

### How to interpret this step

The baseline is 2000–2015 by default. WorldCover 2021 classes form the comparison groups for all years. The reference is the **maximum**, not a percentile. Land cover alone is a simplified grouping, rather than a land-cover-plus-environment stratification. `nppmax` in inherited names refers to the NDVI reference; no NPP model is calculated. Ratio < 0.5 is flagged as low performance. Three-class labels describe relative performance levels, not temporal improvement or decline. Applying a 2021 map to earlier years assumes suitable class stability.


## 1. Import packages

Use the Digital Earth Africa Sandbox for steps that access the data cube. Local raster processing uses the packages listed in `environment.yml`.


In [ ]:
from __future__ import annotations

# =========================================================
# IMPORTS
# =========================================================
import os
import re
import gc
import glob
import time
import warnings
from typing import Dict, List, Tuple, Optional

import numpy as np
import xarray as xr
import rasterio
import geopandas as gpd
import datacube
import rioxarray as rxr
from rasterio.enums import Resampling

from odc.geo.geom import Geometry
from deafrica_tools.areaofinterest import define_area
from deafrica_tools.spatial import xr_rasterize

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


## 2. Load shared settings

The settings below describe the run. Edit the JSON configuration before continuing; later parameter cells explain the settings used by this notebook.


In [ ]:
# Edit workflow_config.json once, then restart and rerun each notebook.
from workflow_support import load_config, require_files, load_aoi, assert_same_grid
cfg = load_config()
print(f"Study: {cfg['study_id']} | Years: {cfg['start_year']}-{cfg['end_year']}")
print(f"Output directory: {cfg['output_dir']}")


## 3. Set reference and assessment periods

Keep ANNUAL_SCALE_FACTOR=1.0 for the floating NDVI written by this workflow. Assessment outputs use one year. Reporting-window outputs exclude the assessment year: 2019 uses 2016–2018, while 2023 uses 2020–2022. Select the appropriate product explicitly during integration.


In [ ]:
# =========================================================
# USER-DEFINED PARAMETERS
# =========================================================
APP_NAME = "performance"

AOI_VECTOR_PATH = cfg["aoi_vector"]
OUTPUT_CRS = cfg["output_crs"]
RESOLUTION = tuple(cfg["resolution"])

ESA_PRODUCT = cfg["esa_product"]
ESA_TIME = cfg["esa_time"]
ESA_MEASUREMENT = "classification"

_OUTPUT = cfg["output_dir"]
ANNUAL_PATTERN = f"{_OUTPUT}/annual_ndvi/ndvi_annual_{cfg['annual_metric']}_*.tif"

OUTPUT_DIR = f"{_OUTPUT}/performance"
PREFIX = "performance"

BASELINE_YEARS = range(cfg["performance_baseline_start"], cfg["performance_baseline_end"] + 1)
THRESHOLD = cfg["performance_threshold"]
LCEU_NODATA_CLASS = 0
ANNUAL_SCALE_FACTOR = 1.0   # use 0.0001 if NDVI stored as scaled integer

# Choose ONE mode:
ASSESSMENT_YEARS = cfg["assessment_years"]   # UNCCD-like reporting
REPORTING_YEARS = None                   # e.g. [2022, 2025]
LOOKBACK_YEARS = 3


## 4. Discover data and align WorldCover

All polygon features are included. WorldCover is reprojected with nearest-neighbour resampling to match the first annual NDVI raster exactly.


In [ ]:
# =========================================================
# MEMORY CLEANUP
# =========================================================
def clear_memory():
    gc.collect()


# =========================================================
# YEAR FROM PATH
# =========================================================
def year_from_path(path: str) -> int:
    base = os.path.basename(path)
    m = re.search(r"(19|20)\d{2}", base)

    if not m:
        raise ValueError(f"Could not find a year in filename: {base}")

    return int(m.group(0))


# =========================================================
# AOI + ESA WORLDCOVER PREP
# =========================================================
def prepare_esa_lceu(
    dc: datacube.Datacube,
    vector_path: str,
    esa_product: str,
    esa_time: str,
    esa_measurement: str,
    output_crs: str,
    resolution: tuple,
    reference_path: str,
) -> xr.Dataset:
    """
    Load ESA WorldCover for AOI and mask to polygon.
    """
    geopolygon_gdf = load_aoi(vector_path)

    lat_range = (
        geopolygon_gdf.total_bounds[1],
        geopolygon_gdf.total_bounds[3],
    )

    lon_range = (
        geopolygon_gdf.total_bounds[0],
        geopolygon_gdf.total_bounds[2],
    )

    query = {
        "x": lon_range,
        "y": lat_range,
        "output_crs": output_crs,
        "resolution": resolution,
    }

    ds_esa = dc.load(
        product=esa_product,
        time=esa_time,
        measurements=[esa_measurement],
        resampling="nearest",
        **query,
    ).squeeze()

    # Match the exact annual pixel grid; categorical classes require nearest-neighbour.
    with rxr.open_rasterio(reference_path, masked=True) as reference:
        ds_esa = ds_esa.rio.reproject_match(reference.squeeze("band", drop=True), resampling=Resampling.nearest)
    mask = xr_rasterize(geopolygon_gdf, ds_esa)
    ds_esa = ds_esa.where(mask)

    clear_memory()
    return ds_esa


## 5. Read annual NDVI and comparison classes

This implementation retains annual arrays in RAM. Missing years, duplicates, and shifted grids are rejected. Large study areas may require tiling or a blockwise redesign.


In [ ]:
# =========================================================
# READ ANNUAL TIFFS
# =========================================================
def read_annual_tif(
    path: str,
    scale_factor: float = 1.0,
) -> Tuple[np.ndarray, dict]:
    """
    Read a single-band annual NDVI GeoTIFF into float32.
    Replace nodata with NaN.
    """
    with rasterio.open(path) as src:
        arr = src.read(1).astype(np.float32)
        profile = src.profile.copy()
        nodata = src.nodata

    if nodata is not None:
        arr = np.where(arr == nodata, np.nan, arr)

    if scale_factor != 1.0:
        arr = arr * float(scale_factor)

    return arr, profile


def build_annual_by_year_from_tiffs(
    annual_tif_paths: List[str],
    scale_factor: float = 1.0,
) -> Tuple[Dict[int, np.ndarray], dict]:
    """
    Returns:
      annual_by_year: {year: annual_ndvi_array}
      ref_profile: profile from first tif
    """
    assert_same_grid(annual_tif_paths)
    annual_by_year = {}
    ref_shape = None
    ref_profile = None

    for path in annual_tif_paths:
        year = year_from_path(path)
        arr, profile = read_annual_tif(path, scale_factor=scale_factor)

        if ref_profile is None:
            ref_profile = profile

        if ref_shape is None:
            ref_shape = arr.shape

        elif arr.shape != ref_shape:
            raise ValueError(
                f"Shape mismatch: {os.path.basename(path)} has shape {arr.shape}, "
                f"expected {ref_shape}. All annual TIFFs must share the same grid."
            )

        if year in annual_by_year:
            raise ValueError(f"Duplicate annual year: {year}")
        annual_by_year[year] = arr

    if ref_profile is None:
        raise ValueError("No annual GeoTIFFs provided.")

    return annual_by_year, ref_profile


# =========================================================
# LCEU FROM ESA WORLDCOVER
# =========================================================
def lceu_from_esa(
    ds_esa: xr.Dataset,
    var_name: str = "classification",
    nodata_class: int = 0,
) -> np.ndarray:
    """
    Convert ESA WorldCover classification to integer LCEU array.
    NaN -> nodata_class.
    """
    if var_name not in ds_esa.data_vars:
        raise ValueError(
            f"'{var_name}' not found. Available: {list(ds_esa.data_vars)}"
        )

    arr = ds_esa[var_name].values

    return np.where(
        np.isfinite(arr),
        arr,
        nodata_class
    ).astype(np.int32)


## 6. Calculate the class reference and performance ratio

For each class, find the maximum of baseline mean NDVI across the AOI and divide the pixel metric by that reference. References ≤0 and invalid pixels become nodata.


In [ ]:
# =========================================================
# BASELINE / PERFORMANCE FUNCTIONS
# =========================================================
def baseline_mean_from_annual(
    annual_by_year: Dict[int, np.ndarray],
    years,
) -> np.ndarray:
    """
    Compute mean NDVI from selected annual rasters.
    """
    years = list(years)
    missing = sorted(set(years) - set(annual_by_year))
    if missing:
        raise ValueError(f"Missing annual inputs for requested mean: {missing}")
    arrs = [
        annual_by_year[y].astype(np.float32)
        for y in years
        if y in annual_by_year
    ]

    if not arrs:
        raise ValueError("No annual NDVI arrays found for requested years.")

    out = np.nanmean(
        np.stack(arrs, axis=0),
        axis=0
    ).astype(np.float32)

    clear_memory()
    return out


def lceu_nppmax_from_metric(
    metric: np.ndarray,
    lceu: np.ndarray,
    nodata_class: int = 0,
) -> np.ndarray:
    """
    For each pixel, assign the maximum metric value found within that LCEU class.
    """
    if metric.shape != lceu.shape:
        raise ValueError(
            f"Shape mismatch: metric {metric.shape} vs LCEU {lceu.shape}"
        )

    out = np.full(metric.shape, np.nan, dtype=np.float32)

    valid = np.isfinite(metric) & (lceu != nodata_class)

    if not np.any(valid):
        raise ValueError("No valid pixels after masking.")

    l_ids = lceu[valid].astype(np.int64)
    m_vals = metric[valid].astype(np.float32)
    flat_valid_idx = np.flatnonzero(valid)

    for uid in np.unique(l_ids):
        idx = l_ids == uid
        out.ravel()[flat_valid_idx[idx]] = np.nanmax(m_vals[idx])

    clear_memory()
    return out


def compute_performance_ratio_binary_and_3class(
    mean_arr: np.ndarray,
    nppmax_lceu: np.ndarray,
    binary_threshold: float = 0.5,
    low_threshold: float = 0.32,
    high_threshold: float = 0.64,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Compute:
      1. Performance ratio
      2. Binary degraded map
      3. Three-class performance map

    Binary degraded:
      0 = Not degraded
      1 = Degraded

    Three-class:
      1 = Low performance     ratio < 0.32
      2 = Moderate performance 0.32 <= ratio <= 0.64
      3 = High performance    ratio > 0.64
    These are relative levels, not temporal trend classes.
    """
    ratio = np.full_like(mean_arr, np.nan, dtype=np.float32)

    ok = (
        np.isfinite(mean_arr)
        & np.isfinite(nppmax_lceu)
        & (nppmax_lceu > 0)
    )

    ratio[ok] = mean_arr[ok] / nppmax_lceu[ok]

    degraded_binary = np.full_like(ratio, np.nan, dtype=np.float32)
    degraded_binary[ok] = (
        ratio[ok] < float(binary_threshold)
    ).astype(np.float32)

    perf_3class = np.full_like(ratio, np.nan, dtype=np.float32)

    perf_3class[ok & (ratio < low_threshold)] = 1
    perf_3class[
        ok & (ratio >= low_threshold) & (ratio <= high_threshold)
    ] = 2
    perf_3class[ok & (ratio > high_threshold)] = 3

    return ratio, degraded_binary, perf_3class


## 7. Define assessment and reporting outputs

Single-year assessments and multi-year reporting windows answer different questions. The output filenames identify which years were averaged.


In [ ]:
# =========================================================
# ASSESSMENT YEAR OUTPUTS
# =========================================================
def compute_assessment_year_outputs(
    annual_by_year: Dict[int, np.ndarray],
    nppmax_lceu: np.ndarray,
    assessment_years,
    binary_threshold: float = 0.5,
    low_threshold: float = 0.32,
    high_threshold: float = 0.64,
) -> Dict[str, np.ndarray]:
    """
    Compute performance ratio, binary degraded map, and 3-class performance map
    for each individual assessment year.

    Example outputs:
      assessment_2015_perf_ratio
      assessment_2015_degraded_binary
      assessment_2015_perf_3class
    """
    out = {}

    if assessment_years is None:
        return out

    for assessment_year in assessment_years:
        assessment_year = int(assessment_year)

        if assessment_year not in annual_by_year:
            raise ValueError(f"Missing assessment-year input: {assessment_year}")

        assessment_arr = annual_by_year[assessment_year].astype(np.float32)

        ratio, degraded_binary, perf_3class = (
            compute_performance_ratio_binary_and_3class(
                assessment_arr,
                nppmax_lceu,
                binary_threshold=binary_threshold,
                low_threshold=low_threshold,
                high_threshold=high_threshold,
            )
        )

        tag = f"assessment_{assessment_year}"

        out[f"{tag}_perf_ratio"] = ratio
        out[f"{tag}_degraded_binary"] = degraded_binary
        out[f"{tag}_perf_3class"] = perf_3class

        assessment_arr = None
        ratio = None
        degraded_binary = None
        perf_3class = None

        clear_memory()

    return out


# =========================================================
# REPORTING WINDOW BUILDERS
# =========================================================
def reporting_windows_from_assessments(
    assessment_years,
    baseline_end_year=2015,
):
    """
    Build reporting windows from assessment years.

    Example:
      baseline_end_year=2015
      assessment_years=[2015, 2019, 2023]

      2019 -> 2016-2018
      2023 -> 2020-2022
    """
    assessment_years = sorted(set(int(y) for y in assessment_years))

    if baseline_end_year not in assessment_years:
        assessment_years = sorted(
            set([baseline_end_year] + assessment_years)
        )

    windows = {}

    for i in range(1, len(assessment_years)):
        prev_year = assessment_years[i - 1]
        current_year = assessment_years[i]

        start = prev_year + 1
        end = current_year - 1

        if start <= end:
            windows[current_year] = list(range(start, end + 1))

    return windows


def reporting_windows_fixed_lookback(
    reporting_years,
    lookback_years=3,
):
    """
    For reporting year R -> use [R-lookback_years, ..., R-1].

    Example:
      R = 2022, lookback = 3 -> 2019, 2020, 2021
    """
    windows = {}

    for r in sorted(set(int(y) for y in reporting_years)):
        start = r - lookback_years
        end = r - 1

        if start <= end:
            windows[r] = list(range(start, end + 1))

    return windows


# =========================================================
# REPORTING OUTPUTS
# =========================================================
def compute_reporting_outputs_for_windows(
    annual_by_year: Dict[int, np.ndarray],
    nppmax_lceu: np.ndarray,
    windows: Dict[int, List[int]],
    binary_threshold: float = 0.5,
    low_threshold: float = 0.32,
    high_threshold: float = 0.64,
) -> Dict[str, np.ndarray]:
    """
    For each reporting year, compute:
      - reporting mean
      - reporting performance ratio
      - reporting degraded binary map
      - reporting 3-class performance map
    """
    out = {}

    for rep_year, years in windows.items():
        missing = sorted(set(years) - set(annual_by_year))
        if missing:
            raise ValueError(f"Missing reporting-window years for {rep_year}: {missing}")
        years_present = list(years)

        if not years_present:
            print(f"No annual data found for reporting window {rep_year}. Skipping.")
            continue

        rep_mean = baseline_mean_from_annual(
            annual_by_year,
            years_present
        )

        rep_ratio, rep_degraded_binary, rep_perf_3class = (
            compute_performance_ratio_binary_and_3class(
                rep_mean,
                nppmax_lceu,
                binary_threshold=binary_threshold,
                low_threshold=low_threshold,
                high_threshold=high_threshold,
            )
        )

        tag = f"rep{rep_year}_{min(years_present)}_{max(years_present)}"

        out[f"{tag}_mean"] = rep_mean
        out[f"{tag}_perf_ratio"] = rep_ratio
        out[f"{tag}_degraded_binary"] = rep_degraded_binary
        out[f"{tag}_perf_3class"] = rep_perf_3class

        rep_mean = None
        rep_ratio = None
        rep_degraded_binary = None
        rep_perf_3class = None

        clear_memory()

    return out


## 8. Write performance rasters

All output rasters, including binary maps, are float32 with −9999 nodata.


In [ ]:
# =========================================================
# WRITE GEOTIFFS
# =========================================================
def write_geotiff(
    path: str,
    array2d: np.ndarray,
    ref_profile: dict,
    nodata_value: float = -9999.0,
    dtype: str = "float32",
):
    """
    Write 2D array to GeoTIFF using reference profile.
    NaN -> nodata_value.
    """
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

    profile = ref_profile.copy()

    profile.update(
        count=1,
        dtype=dtype,
        nodata=nodata_value,
        compress="deflate",
        predictor=2,
    )

    out = array2d.astype(np.float32)
    out = np.where(
        np.isfinite(out),
        out,
        nodata_value
    ).astype(np.float32)

    with rasterio.open(path, "w", **profile) as dst:
        dst.write(out.astype(dtype), 1)


def save_performance_outputs(
    results: Dict[str, np.ndarray],
    ref_profile: dict,
    out_dir: str,
    prefix: str = "perf",
    nodata_value: float = -9999.0,
) -> Dict[str, str]:
    """
    Save all arrays in results dict as GeoTIFFs.
    """
    os.makedirs(out_dir, exist_ok=True)

    output_paths = {}

    for name, arr in results.items():
        out_path = os.path.join(
            out_dir,
            f"{prefix}_{name}.tif"
        )

        write_geotiff(
            out_path,
            arr,
            ref_profile,
            nodata_value=nodata_value,
        )

        output_paths[name] = out_path

    clear_memory()
    return output_paths


## 9. Assemble the performance workflow

The baseline reference is fixed while assessment and reporting metrics vary. Maxima are sensitive to outliers and the AOI boundary. All requested baseline and reporting years must exist.


In [ ]:
# =========================================================
# MAIN DRIVER
# =========================================================
def performance_from_annual_tifs_and_esa_to_geotiff(
    esa_ds: xr.Dataset,
    annual_tif_paths: List[str],
    out_dir: str,
    baseline_years=range(2000, 2016),
    binary_threshold: float = 0.5,
    low_threshold: float = 0.32,
    high_threshold: float = 0.64,
    lceu_nodata_class: int = 0,
    annual_scale_factor: float = 1.0,
    prefix: str = "perf",
    assessment_years=None,
    reporting_years=None,
    lookback_years: int = 3,
):
    """
    Compute productivity performance and save outputs.

    Outputs include:
      - baseline mean
      - baseline LCEU NPPmax
      - baseline performance ratio
      - baseline degraded binary map
      - baseline 3-class performance map
      - assessment-year performance ratio
      - assessment-year degraded binary map
      - assessment-year 3-class performance map
      - optional reporting-window outputs
    """

    baseline_years = list(baseline_years)
    if assessment_years is not None and reporting_years is not None:
        raise ValueError("Choose assessment_years or reporting_years, not both")
    if "classification" not in esa_ds.data_vars:
        raise ValueError(
            f"'classification' not found. Available: {list(esa_ds.data_vars)}"
        )

    # -----------------------------
    # LCEU
    # -----------------------------
    lceu = lceu_from_esa(
        esa_ds,
        var_name="classification",
        nodata_class=lceu_nodata_class,
    )

    # -----------------------------
    # Annual NDVI stack by year
    # -----------------------------
    annual_by_year, ref_profile = build_annual_by_year_from_tiffs(
        annual_tif_paths,
        scale_factor=annual_scale_factor,
    )

    any_year = next(iter(annual_by_year))

    if annual_by_year[any_year].shape != lceu.shape:
        raise ValueError(
            f"Grid mismatch: annual NDVI shape {annual_by_year[any_year].shape} "
            f"!= ESA LCEU shape {lceu.shape}"
        )

    # -----------------------------
    # Baseline performance
    # -----------------------------
    baseline_mean = baseline_mean_from_annual(
        annual_by_year,
        baseline_years,
    )

    nppmax_lceu = lceu_nppmax_from_metric(
        baseline_mean,
        lceu,
        nodata_class=lceu_nodata_class,
    )

    (
        baseline_perf_ratio,
        baseline_degraded_binary,
        baseline_perf_3class,
    ) = compute_performance_ratio_binary_and_3class(
        baseline_mean,
        nppmax_lceu,
        binary_threshold=binary_threshold,
        low_threshold=low_threshold,
        high_threshold=high_threshold,
    )

    results = {
        f"baseline_mean_{min(baseline_years)}_{max(baseline_years)}": baseline_mean,
        "baseline_nppmax_lceu": nppmax_lceu,
        "baseline_perf_ratio": baseline_perf_ratio,
        "baseline_degraded_binary": baseline_degraded_binary,
        "baseline_perf_3class": baseline_perf_3class,
    }

    # -----------------------------
    # Assessment year outputs
    # -----------------------------
    assessment_outputs = compute_assessment_year_outputs(
        annual_by_year=annual_by_year,
        nppmax_lceu=nppmax_lceu,
        assessment_years=assessment_years,
        binary_threshold=binary_threshold,
        low_threshold=low_threshold,
        high_threshold=high_threshold,
    )

    results.update(assessment_outputs)

    # -----------------------------
    # Reporting windows
    # -----------------------------
    if assessment_years is not None:
        windows = reporting_windows_from_assessments(
            assessment_years=assessment_years,
            baseline_end_year=max(baseline_years),
        )

    elif reporting_years is not None:
        windows = reporting_windows_fixed_lookback(
            reporting_years=reporting_years,
            lookback_years=lookback_years,
        )

    else:
        windows = {}

    # -----------------------------
    # Reporting-window outputs
    # -----------------------------
    rep_outputs = compute_reporting_outputs_for_windows(
        annual_by_year=annual_by_year,
        nppmax_lceu=nppmax_lceu,
        windows=windows,
        binary_threshold=binary_threshold,
        low_threshold=low_threshold,
        high_threshold=high_threshold,
    )

    results.update(rep_outputs)

    # -----------------------------
    # Save outputs
    # -----------------------------
    outputs = save_performance_outputs(
        results=results,
        ref_profile=ref_profile,
        out_dir=out_dir,
        prefix=prefix,
        nodata_value=-9999.0,
    )

    # -----------------------------
    # Cleanup
    # -----------------------------
    lceu = None
    annual_by_year = None
    baseline_mean = None
    nppmax_lceu = None
    baseline_perf_ratio = None
    baseline_degraded_binary = None
    baseline_perf_3class = None
    assessment_outputs = None
    rep_outputs = None

    clear_memory()

    return results, outputs


## 10. Load WorldCover and run performance

Confirm that your data cube contains the configured WorldCover product. This notebook overwrites performance outputs; use a fresh output directory for a changed method or study.


In [ ]:
# =========================================================
# RUN WORKFLOW
# =========================================================
dc = datacube.Datacube(app=APP_NAME)

start = time.time()

# -----------------------------
# Discover annual NDVI files
# -----------------------------
annual_tif_paths = sorted(glob.glob(ANNUAL_PATTERN))
if not annual_tif_paths:
    raise ValueError(f"No annual NDVI files found for pattern: {ANNUAL_PATTERN}")

print(f"Found {len(annual_tif_paths)} annual TIFF files")

assert_same_grid(annual_tif_paths)

# -----------------------------
# Prepare ESA WorldCover / LCEU
# -----------------------------
ds_esa_2021 = prepare_esa_lceu(
    dc=dc,
    vector_path=AOI_VECTOR_PATH,
    esa_product=ESA_PRODUCT,
    esa_time=ESA_TIME,
    esa_measurement=ESA_MEASUREMENT,
    output_crs=OUTPUT_CRS,
    resolution=RESOLUTION,
    reference_path=annual_tif_paths[0],
)

# -----------------------------
# Run performance workflow
# -----------------------------
results, outputs = performance_from_annual_tifs_and_esa_to_geotiff(
    esa_ds=ds_esa_2021,
    annual_tif_paths=annual_tif_paths,
    out_dir=OUTPUT_DIR,
    baseline_years=BASELINE_YEARS,
    binary_threshold=THRESHOLD,
    low_threshold=cfg["performance_low_threshold"],
    high_threshold=cfg["performance_high_threshold"],
    annual_scale_factor=ANNUAL_SCALE_FACTOR,
    lceu_nodata_class=LCEU_NODATA_CLASS,
    prefix=PREFIX,
    reporting_years=REPORTING_YEARS,
    lookback_years=LOOKBACK_YEARS,
    assessment_years=ASSESSMENT_YEARS,
)

# -----------------------------
# Final cleanup
# -----------------------------
annual_tif_paths = None
ds_esa_2021 = None
clear_memory()

elapsed = (time.time() - start) / 60

print("\n" + "=" * 60)
print("PROCESS COMPLETED")
print("=" * 60)
print(f"Outputs saved to: {OUTPUT_DIR}")
print(f"Total outputs: {len(outputs)}")
print(f"Elapsed time: {elapsed:.2f} minutes")

for key, value in outputs.items():
    print(f"{key}: {value}")


## Acknowledgements

We gratefully acknowledge the [Group on Earth Observations Land Degradation Neutrality (GEO-LDN) initiative](https://geo-ldn.org/) for its support of this research and the development of this land productivity monitoring workflow.

We acknowledge **Digital Earth Africa (DE Africa)** for providing the open satellite data, analytical tools, and Sandbox platform used in this work.

We also acknowledge the [University of Energy and Natural Resources (UENR), Ghana](https://uenr.edu.gh/), for its support of this research.


## Check the outputs before continuing

- Review warnings and errors, and confirm that every requested year or reporting period was written.
- Open an output raster and inspect its CRS, extent, nodata mask, and value range.
- Keep the configuration and software versions with your results. Run `python scripts/capture_environment.py` from the repository root.
- If you change the AOI, dates, metric, or thresholds, use a fresh output directory to avoid reusing older results.

See [README.md](README.md), [data requirements](docs/DATA_REQUIREMENTS.md), and [method details](docs/METHODS.md).

The supplied notebooks identify their code as Apache License 2.0; retain the original Digital Earth Africa attribution when reusing it. Dataset licences and acknowledgements apply separately. Use the Issues tab of the repository hosting this workflow for workflow-specific questions.

**Validation status:** documentation, syntax, and synthetic-data checks are described in `docs/VALIDATION.md`. This revised workflow has not been executed against the live Digital Earth Africa data cube. A displayed current date or package version is not evidence of a successful full run.
